In [ ]:
##### 01 - inputting required models #####
%matplotlib inline
### setting work directories
import os
import sys
## adding the root directory to the library directory so still can read utility functions
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root) # setting work directory

### libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, r2_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import HistGradientBoostingRegressor
import shap
import joblib
from pathlib import Path
from scipy.stats import pearsonr
import copy
import tqdm

### visualization
import shap
import matplotlib.pyplot as plt
shap.initjs()

## Data loading

In [ ]:
### MESA data
MESA_soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz", 
                        sep = "\t", compression = "gzip").drop(columns={"bmi"})

MESA_olink = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", 
                         sep = "\t", compression = "gzip")

In [ ]:
# indexing
MESA_soma = MESA_soma.set_index("id")
MESA_olink = MESA_olink.set_index("id")

In [ ]:
## protlist
df_prot = pd.read_csv("Results/feature_imp/protlists/olink_soma_selected.tsv", sep = "\t")
protlist = df_prot["x"]


In [ ]:
### test index
test_index = joblib.load("Data/helper_data/test_index.pkl")
MESA_soma_test = MESA_soma.loc[test_index]
MESA_olink_test = MESA_olink.loc[test_index]

In [ ]:
### train index
train_index = joblib.load("Data/helper_data/train_index.pkl")
MESA_soma_train = MESA_soma.loc[train_index]
MESA_olink_train = MESA_olink.loc[train_index]

In [ ]:
olink_prots = MESA_olink.columns

In [ ]:
for protein_name in protlist:
    
    print(f'processing protein: {protein_name}')
    
    protein_model = joblib.load(f'Data/soma_trained_models/olink_soma/3kolink_5k_log2scaled_soma/{protein_name}_hgb.pkl')
    y_true = MESA_soma_test[f'{protein_name}']
    
    print(f'processing protein: {protein_name}, starting to do PFI')
    # permutation feature importance
    result = permutation_importance(
        protein_model, MESA_olink_test, y_true, 
        n_repeats=5,
        random_state=613, 
        n_jobs=-1
    )
    
    print(f'processing protein: {protein_name}, putting things in dataframe')
    
    importance_df = pd.DataFrame({
        'feature': MESA_olink_test.columns,
        'importance_mean': result.importances_mean,
        'importance_std': result.importances_std
    }).sort_values(by='importance_mean', ascending=False)
    
    importance_df.to_csv(f"Results/feature_imp/olink_soma/{protein_name}_feature_imp.tsv", index=False, sep="\t")
    
    
    
    print(f'processing protein: {protein_name}, scoring r2')

    rank = np.argsort(result.importances_mean)[::-1]
    rank
    ks = [1,2,3,4,5,7,10,15,20,30,40,50,70,100,150,200,300]
    model_scores = []
    corr_scores = []

    if protein_name in olink_prots:
        for k in ks:
            feats = rank[:k]
            model_k = HistGradientBoostingRegressor(random_state=0)
            model_k.fit(MESA_olink_train.iloc[:, feats], MESA_soma_train[f'{protein_name}'])
            model_scores.append(
                pearsonr(model_k.predict(MESA_olink_test.iloc[:, feats]), MESA_soma_test[f'{protein_name}'])[0]
            )
            corr_scores.append(
                pearsonr(model_k.predict(MESA_olink_test.iloc[:, feats]), MESA_olink_test[f'{protein_name}'])[0]
            )


        ## saving score dataframe
        score_df = pd.DataFrame({
        "k": ks,
        "model_score": model_scores,
        "corr_score": corr_scores
        })

        score_df.to_csv(
            f"Results/feature_imp/olink_soma/{protein_name}_ablation_r2.tsv", index=False, sep="\t")

    else:
        for k in ks:
            feats = rank[:k]
            model_k = HistGradientBoostingRegressor(random_state=0)
            model_k.fit(MESA_olink_train.iloc[:, feats], MESA_soma_train[f'{protein_name}'])
            model_scores.append(
                pearsonr(model_k.predict(MESA_olink_test.iloc[:, feats]), MESA_soma_test[f'{protein_name}'])[0]
            )
    #             corr_scores.append(
    #                 pearsonr(model_k.predict(MESA_olink_test.iloc[:, feats]), MESA_olink_test[f'{protein_name}'])[0]
    #             )

        ## saving score dataframe
        score_df = pd.DataFrame({
        "k": ks,
        "model_score": model_scores,
        "corr_score": np.nan
        })

        score_df.to_csv(
            f"Results/feature_imp/olink_soma/{protein_name}_ablation_r2.tsv", index=False, sep="\t")